In [ ]:
import json
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict
import numpy as np

In [5]:
import json
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict
import numpy as np

def prepare_dfdc_data_splits(
    source_data_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data",
    output_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized",
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    random_state=42
):
    """
    Prepare DFDC data in the required format:
    data/
      train/real_videos/*.mp4
      train/fake_videos/*.mp4
      val/real_videos/*.mp4
      val/fake_videos/*.mp4
      test/real_videos/*.mp4
      test/fake_videos/*.mp4
    
    Args:
        source_data_dir: Path to the original DFDC data directory
        output_dir: Path where organized data will be saved
        train_ratio: Proportion of data for training (default: 0.8)
        val_ratio: Proportion of data for validation (default: 0.1)
        test_ratio: Proportion of data for testing (default: 0.1)
        random_state: Random seed for reproducible splits
    """
    
    # Verify ratios sum to 1
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"
    
    # Paths
    source_videos_dir = Path(source_data_dir) / "train_sample_videos"
    metadata_path = source_videos_dir / "metadata.json"
    output_path = Path(output_dir)
    
    # Load metadata
    print(f"Loading metadata from: {metadata_path}")
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    # Organize videos by label
    real_videos = []
    fake_videos = []
    
    for video_file, info in metadata.items():
        if info['label'] == 'REAL':
            real_videos.append(video_file)
        elif info['label'] == 'FAKE':
            fake_videos.append(video_file)
    
    print(f"Found {len(real_videos)} REAL videos")
    print(f"Found {len(fake_videos)} FAKE videos")
    
    # Function to split data stratified by label
    def stratified_split(videos, train_r, val_r, test_r, random_state):
        # First split: train vs (val + test)
        train_videos, temp_videos = train_test_split(
            videos, 
            test_size=(val_r + test_r), 
            random_state=random_state
        )
        
        # Second split: val vs test from the temp set
        val_videos, test_videos = train_test_split(
            temp_videos,
            test_size=test_r / (val_r + test_r),  # Adjust ratio for the remaining data
            random_state=random_state
        )
        
        return train_videos, val_videos, test_videos
    
    # Split real and fake videos separately to maintain stratification
    real_train, real_val, real_test = stratified_split(
        real_videos, train_ratio, val_ratio, test_ratio, random_state
    )
    fake_train, fake_val, fake_test = stratified_split(
        fake_videos, train_ratio, val_ratio, test_ratio, random_state
    )
    
    # Print split statistics
    print(f"\\nSplit Statistics:")
    print(f"REAL videos - Train: {len(real_train)}, Val: {len(real_val)}, Test: {len(real_test)}")
    print(f"FAKE videos - Train: {len(fake_train)}, Val: {len(fake_val)}, Test: {len(fake_test)}")
    print(f"Total - Train: {len(real_train) + len(fake_train)}, Val: {len(real_val) + len(fake_val)}, Test: {len(real_test) + len(fake_test)}")
    
    # Create directory structure
    splits = {
        'train': {'real_videos': real_train, 'fake_videos': fake_train},
        'val': {'real_videos': real_val, 'fake_videos': fake_val},
        'test': {'real_videos': real_test, 'fake_videos': fake_test}
    }
    
    # Create directories and copy files
    for split_name, categories in splits.items():
        for category, video_list in categories.items():
            target_dir = output_path / split_name / category
            target_dir.mkdir(parents=True, exist_ok=True)
            
            print(f"\\nCopying {len(video_list)} videos to {target_dir}")
            
            for video_file in video_list:
                source_file = source_videos_dir / video_file
                target_file = target_dir / video_file
                
                if source_file.exists():
                    shutil.copy2(source_file, target_file)
                else:
                    print(f"Warning: Source file not found: {source_file}")
    
    # Create summary metadata for each split
    summary = {
        'splits': {
            'train': {
                'real_count': len(real_train),
                'fake_count': len(fake_train),
                'total': len(real_train) + len(fake_train)
            },
            'val': {
                'real_count': len(real_val),
                'fake_count': len(fake_val),
                'total': len(real_val) + len(fake_val)
            },
            'test': {
                'real_count': len(real_test),
                'fake_count': len(fake_test),
                'total': len(real_test) + len(fake_test)
            }
        },
        'ratios': {
            'train': train_ratio,
            'val': val_ratio,
            'test': test_ratio
        },
        'random_state': random_state
    }
    
    # Save summary
    summary_path = output_path / 'data_split_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\\nData organization complete!")
    print(f"Organized data saved to: {output_path}")
    print(f"Summary saved to: {summary_path}")
    
    return summary

# Run the data preparation
summary = prepare_dfdc_data_splits()
print("\\nFinal Summary:")
print(json.dumps(summary, indent=2))


Loading metadata from: /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/train_sample_videos/metadata.json
Found 77 REAL videos
Found 323 FAKE videos
\nSplit Statistics:
REAL videos - Train: 61, Val: 8, Test: 8
FAKE videos - Train: 258, Val: 32, Test: 33
Total - Train: 319, Val: 40, Test: 41
\nCopying 61 videos to /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized/train/real_videos
\nCopying 258 videos to /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized/train/fake_videos
\nCopying 8 videos to /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized/val/real_videos
\nCopying 32 videos to /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized/val/fake_videos
\nCopying 8 videos to /Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized/test/real_videos
\nCopying 33 videos to /Users/crownedprinz/Documents/Proje

In [6]:
# Additional utility functions for working with the organized data

def load_organized_data_info(organized_data_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized"):
    """
    Load information about the organized data splits
    """
    organized_path = Path(organized_data_dir)
    summary_path = organized_path / 'data_split_summary.json'
    
    if summary_path.exists():
        with open(summary_path, 'r') as f:
            summary = json.load(f)
        return summary
    else:
        print(f"Summary file not found at {summary_path}")
        return None

def get_video_paths_by_split(organized_data_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized"):
    """
    Get all video paths organized by split and label
    
    Returns:
        dict: Nested dictionary with structure:
              {split: {label: [list_of_video_paths]}}
    """
    organized_path = Path(organized_data_dir)
    video_paths = {}
    
    for split in ['train', 'val', 'test']:
        video_paths[split] = {}
        for label in ['real_videos', 'fake_videos']:
            split_dir = organized_path / split / label
            if split_dir.exists():
                video_paths[split][label] = list(split_dir.glob('*.mp4'))
            else:
                video_paths[split][label] = []
    
    return video_paths

def create_dataframe_from_organized_data(organized_data_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized"):
    """
    Create a pandas DataFrame with video paths and labels for easy manipulation
    
    Returns:
        pd.DataFrame: DataFrame with columns ['video_path', 'label', 'split', 'filename']
    """
    video_paths = get_video_paths_by_split(organized_data_dir)
    
    data = []
    for split, labels in video_paths.items():
        for label_dir, paths in labels.items():
            # Convert label_dir to simple label
            label = 'REAL' if label_dir == 'real_videos' else 'FAKE'
            
            for path in paths:
                data.append({
                    'video_path': str(path),
                    'label': label,
                    'split': split,
                    'filename': path.name
                })
    
    return pd.DataFrame(data)

def verify_data_organization(organized_data_dir="/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized"):
    """
    Verify the data organization and print statistics
    """
    print("Verifying data organization...")
    
    # Load summary
    summary = load_organized_data_info(organized_data_dir)
    if summary:
        print("\\nSummary from metadata:")
        for split, counts in summary['splits'].items():
            print(f"  {split.upper()}: {counts['real_count']} real, {counts['fake_count']} fake, {counts['total']} total")
    
    # Count actual files
    video_paths = get_video_paths_by_split(organized_data_dir)
    print("\\nActual file counts:")
    
    total_real = 0
    total_fake = 0
    
    for split, labels in video_paths.items():
        real_count = len(labels['real_videos'])
        fake_count = len(labels['fake_videos'])
        total_count = real_count + fake_count
        
        total_real += real_count
        total_fake += fake_count
        
        print(f"  {split.upper()}: {real_count} real, {fake_count} fake, {total_count} total")
    
    print(f"\\nOverall totals: {total_real} real, {total_fake} fake, {total_real + total_fake} total")
    
    # Create and display DataFrame
    df = create_dataframe_from_organized_data(organized_data_dir)
    print(f"\\nDataFrame shape: {df.shape}")
    print("\\nDataFrame sample:")
    print(df.head())
    
    print("\\nLabel distribution by split:")
    print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
    
    return df

# Run verification if the organized data exists
organized_dir = "/Users/crownedprinz/Documents/Projects/Python/Deepfake-Detector/model/data/organized"
if Path(organized_dir).exists():
    df = verify_data_organization(organized_dir)
else:
    print(f"Organized data directory not found: {organized_dir}")
    print("Run the first cell to organize the data first.")


Verifying data organization...
\nSummary from metadata:
  TRAIN: 61 real, 258 fake, 319 total
  VAL: 8 real, 32 fake, 40 total
  TEST: 8 real, 33 fake, 41 total
\nActual file counts:
  TRAIN: 61 real, 258 fake, 319 total
  VAL: 8 real, 32 fake, 40 total
  TEST: 8 real, 33 fake, 41 total
\nOverall totals: 77 real, 323 fake, 400 total
\nDataFrame shape: (400, 4)
\nDataFrame sample:
                                          video_path label  split  \
0  /Users/crownedprinz/Documents/Projects/Python/...  REAL  train   
1  /Users/crownedprinz/Documents/Projects/Python/...  REAL  train   
2  /Users/crownedprinz/Documents/Projects/Python/...  REAL  train   
3  /Users/crownedprinz/Documents/Projects/Python/...  REAL  train   
4  /Users/crownedprinz/Documents/Projects/Python/...  REAL  train   

         filename  
0  eudeqjhdfd.mp4  
1  cizlkenljw.mp4  
2  eckvhdusax.mp4  
3  bpapbctoao.mp4  
4  caifxvsozs.mp4  
\nLabel distribution by split:
label  FAKE  REAL
split            
test     33    